In [10]:
import numpy as np
import pandas as pd

In [11]:
dtype_map = {
    "Task Id": "string",
    "FPOGV": "Int64", "FPOGID": "Int64",
    "LPMMV": "Int64", "RPMMV": "Int64",
    "BKID": "Int64",  "BKDUR": "float64", 
    "AOI": "string"
}
fixations_onoff = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/fix_onoff_task1.csv", dtype=dtype_map)
gaze_onoff = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/gaze_onoff_task1.csv", dtype=dtype_map)

In [13]:
# ============================================
# Shared setup (windowing, helpers) — as given
# ============================================
import numpy as np
import pandas as pd

# Gazepoint TIMETICK frequency
TICK_HZ = 10_000_000  # TIMETICK(f=10000000)

# -------------------------------------------------------
# A) Windowing & label refinement
# -------------------------------------------------------
def build_shared_anchor(
    *dfs,
    tick_col: str = "TIMETICK(f=10000000)",
    group_cols=("Video Name (folder)", "User ID (file)")
) -> pd.DataFrame:
    refs = []
    for d in dfs:
        cols = list(group_cols) + [tick_col]
        tmp = d[cols].copy()
        tmp[tick_col] = pd.to_numeric(tmp[tick_col], errors="coerce")
        refs.append(
            tmp.groupby(list(group_cols), as_index=False)[tick_col]
               .min()
               .rename(columns={tick_col: "min_tick"})
        )
    ref = (
        pd.concat(refs, ignore_index=True)
          .groupby(list(group_cols), as_index=False)["min_tick"].min()
          .rename(columns={"min_tick": "anchor_tick"})
    )
    return ref

def add_window_id_by_timetick(
    df: pd.DataFrame,
    *,
    win_sec: float,
    tick_col: str = "TIMETICK(f=10000000)",
    group_cols=("Video Name (folder)", "User ID (file)"),
    shared_ref: pd.DataFrame | None = None,
    out_col: str = "win_id",
) -> pd.DataFrame:
    out = df.copy()
    out[tick_col] = pd.to_numeric(out[tick_col], errors="coerce")
    win_ticks = int(round(win_sec * TICK_HZ))

    if shared_ref is not None:
        out = out.merge(shared_ref, on=list(group_cols), how="left", validate="many_to_one")
        anchor = out["anchor_tick"]
    else:
        anchor = out.groupby(list(group_cols))[tick_col].transform("min")

    out[out_col] = ((out[tick_col] - anchor) // win_ticks).astype("Int64")
    return out

def refine_window_labels_by_fraction(
    df_with_win: pd.DataFrame,
    *,
    group_cols=("Video Name (folder)", "User ID (file)"),
    win_col="win_id",
    label_col="label",
    on_hi=0.80,
    off_lo=0.20,
    out_col="label",
) -> pd.DataFrame:
    d = df_with_win.copy()
    d[label_col] = d[label_col].astype(str).str.upper().str.strip()
    d = d[d[label_col].isin(["ON", "OFF"])].copy()

    key = list(group_cols) + [win_col]
    d["_is_on"] = (d[label_col] == "ON").astype(float)

    frac = (
        d.groupby(key, dropna=False)["_is_on"].mean()
         .rename("on_fraction").reset_index()
    )

    frac["label_refined"] = np.where(
        frac["on_fraction"] >= on_hi, "ON",
        np.where(frac["on_fraction"] <= off_lo, "OFF", np.nan)
    )

    d = d.merge(frac[key + ["label_refined", "on_fraction"]],
                on=key, how="left", validate="many_to_one")

    d = d[~d["label_refined"].isna()].copy()
    d[out_col] = d["label_refined"]
    d.drop(columns=["_is_on", "label_refined"], inplace=True)
    return d

# -------------------------------------------------------
# B) Shared feature helpers
# -------------------------------------------------------
def _mad(x):
    x = pd.Series(x, dtype="float64").dropna()
    if x.empty: return np.nan
    med = x.median()
    return (x - med).abs().median()

def _pupil_slope(tick, pupil):
    if tick is None:
        return np.nan
    tt = pd.Series(tick, dtype="float64")
    pp = pd.Series(pupil, dtype="float64")
    m = (~tt.isna()) & (~pp.isna())
    if m.sum() < 5:
        return np.nan
    tsec = (tt[m] - tt[m].iloc[0]) / TICK_HZ
    try:
        slope, _ = np.polyfit(tsec.values, pp[m].values, 1)
    except Exception:
        slope = np.nan
    return slope

def _circular_variance_deg(angles_deg):
    a = pd.Series(angles_deg, dtype="float64").dropna()
    if a.empty: return np.nan
    rad = np.deg2rad(a.values)
    C = np.mean(np.cos(rad))
    S = np.mean(np.sin(rad))
    R = np.sqrt(C*C + S*S)
    return 1 - R

def _valid_mask(w: pd.DataFrame) -> pd.Series:
    return (w.get("FPOGV", 1) == 1)

def _safe_std(s: pd.Series) -> float:
    s = pd.to_numeric(s, errors="coerce").dropna()
    return float(s.std()) if len(s) else np.nan

def _gaze_entropy_2d(x, y, *, bins=8, min_valid=100):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < min_valid or y.size < min_valid:
        return np.nan
    x = np.clip(x, 0.0, 1.0)
    y = np.clip(y, 0.0, 1.0)
    H, _, _ = np.histogram2d(x, y, bins=bins, range=[[0,1],[0,1]], density=False)
    total = H.sum()
    if total <= 0:
        return np.nan
    p = (H / total).ravel()
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def _pupil_series(w: pd.DataFrame) -> pd.Series:
    """Mean of valid eyes per sample; NaN if none valid."""
    lv = (w.get("LPMMV", 0) == 1)
    rv = (w.get("RPMMV", 0) == 1)
    l = w["LPMM"].where(lv) if "LPMM" in w else None
    r = w["RPMM"].where(rv) if "RPMM" in w else None
    if l is not None and r is not None:
        return pd.concat([l, r], axis=1).mean(axis=1, skipna=True)
    if l is not None:
        return l
    if r is not None:
        return r
    return pd.Series(index=w.index, dtype="float64")

def _approx_sample_rate_hz(ticks: pd.Series) -> float:
    t = pd.to_numeric(ticks, errors="coerce").dropna().values
    if t.size < 3:
        return np.nan
    dt = np.diff(t) / TICK_HZ
    med = np.median(dt[dt > 0])
    return float(1.0 / med) if med > 0 else np.nan

# -------------------------------------------------------
# C1) Base windowed features (fixations / gaze) — as in Task 1
# -------------------------------------------------------
def window_features_fixations(fix_df: pd.DataFrame, *, win_sec: float) -> pd.DataFrame:
    keys = ["Video Name (folder)", "User ID (file)", "Task Id", "type", "label", "phase", "block_id", "win_id"]
    eps = 1e-6

    def _agg(w: pd.DataFrame) -> pd.Series:
        fix_count      = w["FPOGID"].nunique() if "FPOGID" in w else np.nan
        fix_mean_dur   = w["FPOGD"].mean()     if "FPOGD"  in w else np.nan
        fix_median_dur = w["FPOGD"].median()   if "FPOGD"  in w else np.nan
        fix_aoi_ratio  = w["AOI"].notna().mean() if "AOI" in w else np.nan

        fix_efficiency = (fix_count / (abs(fix_mean_dur) + eps)) if pd.notna(fix_mean_dur) else np.nan
        fix_rate_per_s = (fix_count / win_sec) if pd.notna(fix_count) else np.nan

        return pd.Series({
            "fix_efficiency":  fix_efficiency,
            "fix_aoi_ratio":   fix_aoi_ratio,
            "fix_rate_per_s":  fix_rate_per_s,
            "fix_median_dur":  fix_median_dur,
        })

    return (
        fix_df.dropna(subset=["win_id"])
              .groupby(keys, dropna=False).apply(_agg).reset_index()
    )

def window_features_gaze(gaze_df: pd.DataFrame, *, win_sec: float = 3.0) -> pd.DataFrame:
    keys = ["Video Name (folder)", "User ID (file)", "Task Id", "type", "label", "phase", "block_id", "win_id"]

    def _agg(w: pd.DataFrame) -> pd.Series:
        gv = (w["FPOGV"] == 1) if "FPOGV" in w else pd.Series(True, index=w.index)

        gx = w.loc[gv, "FPOGX"].std() if "FPOGX" in w else np.nan
        gy = w.loc[gv, "FPOGY"].std() if "FPOGY" in w else np.nan
        gaze_disp_total = (
            np.sqrt((0 if pd.isna(gx) else gx**2) + (0 if pd.isna(gy) else gy**2))
            if (not pd.isna(gx) or not pd.isna(gy)) else np.nan
        )

        pup = _pupil_series(w)
        if pup.notna().any():
            pupil_mean  = float(pup.mean())
            pupil_var   = float(pup.var(ddof=1)) if pup.count() > 1 else np.nan
            pupil_mad   = _mad(pup)
            tick_col = "TIMETICK(f=10000000)"
            pupil_slope = _pupil_slope(w[tick_col] if tick_col in w else None, pup)
        else:
            pupil_mean = pupil_var = pupil_mad = pupil_slope = np.nan

        saccade_mean_mag = w.loc[gv, "SACCADE_MAG"].mean() if "SACCADE_MAG" in w else np.nan
        if "SACCADE_DIR" in w:
            saccade_circ_var = _circular_variance_deg(w.loc[gv, "SACCADE_DIR"])
            saccade_dir_var  = w.loc[gv, "SACCADE_DIR"].var(ddof=1)
        else:
            saccade_circ_var = np.nan
            saccade_dir_var  = np.nan
        if pd.isna(saccade_dir_var):
            saccade_dir_var = 0.0

        blink_count      = w.loc[gv, "BKID"].nunique() if "BKID" in w else np.nan
        blink_rate       = (blink_count / win_sec) if pd.notna(blink_count) else np.nan
        valid_gaze_ratio = float(gv.mean()) if len(w) else np.nan

        return pd.Series({
            "gaze_dispersion_total": gaze_disp_total,
            "pupil_mean":            pupil_mean,
            "pupil_var":             0 if pd.isna(pupil_var) else pupil_var,
            "pupil_mad":             pupil_mad,
            "pupil_slope":           pupil_slope,
            "saccade_mean_mag":      saccade_mean_mag,
            "saccade_dir_var":       0 if pd.isna(saccade_dir_var) else saccade_dir_var,
            "saccade_circ_var":      saccade_circ_var,
            "blink_rate":            blink_rate,
            "valid_gaze_ratio":      valid_gaze_ratio,
        })

    return (
        gaze_df.dropna(subset=["win_id"])
               .groupby(keys, dropna=False).apply(_agg).reset_index()
    )

# -------------------------------------------------------
# C2) Fade-specific windowed features (5 items)
# -------------------------------------------------------
def window_features_fade(
    gaze_df: pd.DataFrame,
    *,
    win_sec: float,
    low_vel_thresh: float = 0.25,   # normalized units/sec; tune if needed
    pupil_roll_ms: int = 400,       # 300–500ms recommended
    entropy_bins: int = 8
) -> pd.DataFrame:
    """
    Fade-specific features per window:
      - low_vel_frac
      - fix_stability             (MAD/median of FPOGD)
      - fixation_entropy_2d       (H)
      - max_aoi_dwell_frac
      - pupil_roll_std_mean
    Assumes FPOGX/FPOGY in [0,1]; uses TIMETICK to compute speeds & rolling windows.
    """
    keys = ["Video Name (folder)", "User ID (file)", "Task Id", "type", "label", "phase", "block_id", "win_id"]
    df = gaze_df.dropna(subset=["win_id"]).copy()

    def _speed_series(w: pd.DataFrame) -> pd.Series:
        vmask = _valid_mask(w)
        if not ({"FPOGX","FPOGY","TIMETICK(f=10000000)"} <= set(w.columns)):
            return pd.Series(dtype="float64")
        x = pd.to_numeric(w.loc[vmask, "FPOGX"], errors="coerce").to_numpy()
        y = pd.to_numeric(w.loc[vmask, "FPOGY"], errors="coerce").to_numpy()
        t = pd.to_numeric(w.loc[vmask, "TIMETICK(f=10000000)"], errors="coerce").to_numpy()
        m = (~np.isnan(x)) & (~np.isnan(y)) & (~np.isnan(t))
        x, y, t = x[m], y[m], t[m]
        if x.size < 2:
            return pd.Series(dtype="float64")
        dx = np.diff(x); dy = np.diff(y); dt = np.diff(t) / TICK_HZ
        dt[dt <= 0] = np.nan
        spd = np.sqrt(dx*dx + dy*dy) / dt
        return pd.Series(spd)

    def _aoi_dwell_frac(w: pd.DataFrame) -> float:
        if "AOI" not in w:
            return np.nan
        a = w["AOI"].astype(str).str.strip()
        mask = ~a.str.lower().isin(["", "none", "null", "nan"])
        a = a[mask]
        if a.empty:
            return np.nan
        frac = a.value_counts(normalize=True)
        return float(frac.max())

    def _pupil_rolling_std_mean(w: pd.DataFrame, ms: int) -> float:
        if "TIMETICK(f=10000000)" not in w:
            return np.nan
        pup = _pupil_series(w)
        if not pup.notna().any():
            return np.nan
        hz = _approx_sample_rate_hz(w["TIMETICK(f=10000000)"])
        if not np.isfinite(hz) or hz <= 0:
            return np.nan
        win = max(int(round(hz * ms / 1000.0)), 2)
        roll = pup.rolling(win, min_periods=max(2, win//2)).std()
        return float(roll.mean(skipna=True))

    def _fix_stability(w: pd.DataFrame) -> float:
        if "FPOGD" not in w:
            return np.nan
        durs = pd.to_numeric(w["FPOGD"], errors="coerce").dropna()
        if durs.empty or np.nanmedian(durs) == 0:
            return np.nan
        return float(_mad(durs) / np.nanmedian(durs))

    def _agg(w: pd.DataFrame) -> pd.Series:
        # Low-velocity fraction
        spd = _speed_series(w)
        low_vel_frac = float((spd < low_vel_thresh).mean()) if spd.size else np.nan

        # Fixation stability
        fix_stability = _fix_stability(w)

        # Spatial entropy
        vmask = _valid_mask(w)
        H = np.nan
        if {"FPOGX","FPOGY"} <= set(w.columns):
            H = _gaze_entropy_2d(w.loc[vmask,"FPOGX"], w.loc[vmask,"FPOGY"], bins=entropy_bins, min_valid=60)

        # AOI dwell concentration
        max_aoi_dwell_frac = _aoi_dwell_frac(w)

        # Pupil rolling std mean
        pupil_roll_std_mean = _pupil_rolling_std_mean(w, pupil_roll_ms)

        return pd.Series({
            "low_vel_frac":          low_vel_frac,
            "fix_stability":         fix_stability,
            "fixation_entropy_2d":   H,
            "max_aoi_dwell_frac":    max_aoi_dwell_frac,
            "pupil_roll_std_mean":   pupil_roll_std_mean,
        })

    return (
        df.groupby(keys, dropna=False).apply(_agg).reset_index()
    )

# -------------------------------------------------------
# D) Merge all features (Fade variant)
# -------------------------------------------------------
def combine_fix_gaze_fade(
    fix_df: pd.DataFrame,
    gaze_df: pd.DataFrame,
    *,
    win_sec: float = 3.0,
    low_vel_thresh: float = 0.25,
    pupil_roll_ms: int = 400,
    entropy_bins: int = 8
) -> pd.DataFrame:
    """
    Compute & merge base (fixation + gaze) features with Fade-specific features.
    Expects input dataframes already contain 'win_id' and task/meta columns.
    """
    fix_w  = window_features_fixations(fix_df, win_sec=win_sec)
    gaze_w = window_features_gaze(gaze_df,    win_sec=win_sec)
    fade_w = window_features_fade(
        gaze_df,
        win_sec=win_sec,
        low_vel_thresh=low_vel_thresh,
        pupil_roll_ms=pupil_roll_ms,
        entropy_bins=entropy_bins,
    )

    keys = ["Video Name (folder)", "User ID (file)", "Task Id", "type", "label", "phase", "block_id", "win_id"]
    m = pd.merge(fix_w,  gaze_w, on=keys, how="outer", validate="one_to_one")
    m = pd.merge(m,      fade_w, on=keys, how="outer", validate="one_to_one")
    return m

# =========================
# Example usage (uncomment & adapt)
# =========================


In [14]:
raw = pd.read_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/outputs/fix_onoff_task3.csv")
win_len = 3.0
ref = build_shared_anchor(raw, tick_col="TIMETICK(f=10000000)")
raw = add_window_id_by_timetick(raw, win_sec=win_len, shared_ref=ref)
raw_refined = refine_window_labels_by_fraction(raw, win_col="win_id", label_col="label", on_hi=0.80, off_lo=0.20)

features_task3 = combine_fix_gaze_fade(raw_refined, raw_refined, win_sec=win_len)
display(features_task3.head())

/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_12076/4024181688.py:194: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .groupby(keys, dropna=False).apply(_agg).reset_index()
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_00000gn/T/ipykernel_12076/4024181688.py:249: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .groupby(keys, dropna=False).apply(_agg).reset_index()
/var/folders/jh/6lrcqwz57w5_2ylmrnwyby_0

,Video Name (folder),User ID (file),Task Id,type,label,phase,block_id,win_id,fix_efficiency,fix_aoi_ratio,...,saccade_mean_mag,saccade_dir_var,saccade_circ_var,blink_rate,valid_gaze_ratio,low_vel_frac,fix_stability,fixation_entropy_2d,max_aoi_dwell_frac,pupil_roll_std_mean
0,P10_S1_ITA,User 0,3.1,FADE,ON,ON,P10_S1_ITA|User 0|3.1,1,27.690246,0.0,...,1402.266638,6815.790855,0.465056,0.666667,1.0,0.250000,0.002484,NaN,NaN,0.118998
1,P10_S1_ITA,User 0,3.1,FADE,nan,ON,P10_S1_ITA|User 0|3.1,2,9.728999,0.0,...,240.676455,25935.525470,0.595240,0.333333,1.0,0.000000,0.040376,NaN,NaN,NaN
2,P10_S1_ITA,User 0,3.1,Unknown,OFF,OFF_POST,P10_S1_ITA|User 0|3.1,3,14.544079,0.0,...,3541.506888,10371.545595,0.957963,0.333333,1.0,0.333333,0.249378,NaN,NaN,0.525801
3,P10_S1_ITA,User 0,3.1,Unknown,OFF,OFF_PRE,P10_S1_ITA|User 0|3.1,0,43.619938,0.0,...,1412.489640,7716.793929,0.675368,0.666667,1.0,0.250000,0.288764,NaN,NaN,0.072765
4,P10_S1_ITA,User 0,3.1,Unknown,ON,OFF_PRE,P10_S1_ITA|User 0|3.1,1,2.255498,0.0,...,15.972170,0.000000,0.000000,0.333333,1.0,NaN,0.000000,NaN,NaN,NaN


In [15]:
features_task3.to_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/improving models/outputs3/fix_wid_t3.csv", index=False)    

In [16]:
# --- your mappings ---
task_mapping = {
    '1.1': 'C011', '1.2': 'C100', '1.3': 'C000', '1.4': 'Cnull', '1.5': 'C100',
    '1.6': 'C110', '1.7': 'C011', '1.8': 'C001', '1.9': 'C111', '2.1': 'B000',
    '2.2': 'Bnull', '2.3': 'B112', '2.4': 'B113', '2.5': 'B012', '2.6': 'B013',
    '2.7': 'B002', '2.8': 'B102', '2.9': 'B103', '3.1': 'A000', '3.2': 'A100',
    '3.3': 'A010', '3.4': 'A101', '3.5': 'A001', '3.6': 'A111', '3.7': 'A110',
    '3.8': 'A011', '3.9': 'Anull'
}

animation_mapping = {
    'A': {
        'type': 'FADE',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': ['Left to Right', 'Top to Bottom'],
    },
    'B': {
        'type': 'BOUNCE',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': {0: 'Left to Right', 1: 'Top to Bottom', 2: 'Bouncing', 3: 'Pulsing'},
    },
    'C': {
        'type': 'SCROLL',
        'delay': ['0.3s', '1s'],
        'ease': ['Ease-in', 'Ease-out'],
        'direction': ['Left to Right', 'Top to Bottom'],
    },
}

def decode_from_code(code: str):
    """
    Return dict with decoded fields from a code like 'C011', 'B112', or '*null'.
    """
    if not isinstance(code, str) or "null" in code.lower():
        return {
            "anim_family": np.nan,
            "anim_code": code if isinstance(code, str) else np.nan,
            "anim_type": "NO-ANIM",
            "anim_delay": np.nan,
            "anim_ease": np.nan,
            "anim_direction": np.nan,
            "block_anim": False,
        }

    fam = code[0]                 # 'A' / 'B' / 'C'
    spec = code[1:]               # e.g. '011' or '112'
    meta = animation_mapping.get(fam, {})

    # defaults
    delay = ease = direction = np.nan

    # parse indices safely
    try:
        i_delay = int(spec[0]) if len(spec) >= 1 else 0
        i_ease  = int(spec[1]) if len(spec) >= 2 else 0
        i_dir   = int(spec[2]) if len(spec) >= 3 else 0
    except ValueError:
        i_delay = i_ease = i_dir = 0

    # map delay/ease
    if "delay" in meta and isinstance(meta["delay"], list) and i_delay < len(meta["delay"]):
        delay = meta["delay"][i_delay]
    if "ease" in meta and isinstance(meta["ease"], list) and i_ease < len(meta["ease"]):
        ease = meta["ease"][i_ease]

    # map direction (B uses dict, A/C use list)
    if fam == "B" and isinstance(meta.get("direction"), dict):
        direction = meta["direction"].get(i_dir, np.nan)
    elif isinstance(meta.get("direction"), list) and i_dir < len(meta["direction"]):
        direction = meta["direction"][i_dir]

    return {
        "anim_family": fam,
        "anim_code": code,
        "anim_type": meta.get("type", np.nan),
        "delay": delay,
        "ease": ease,
        "direction": direction,
        "anim": True,
    }

def add_animation_columns(windows_df: pd.DataFrame) -> pd.DataFrame:
    """
    Make a copy of windows_df and add:
      anim_family, anim_code, anim_type, anim_delay, anim_ease, anim_direction, block_anim
    based on Task Id -> code mapping.
    """
    out = windows_df.copy()

    # ensure string task ids for mapping
    out["Task Id"] = out["Task Id"].astype(str).str.strip()

    # look up the code for each Task Id
    out["anim_code"] = out["Task Id"].map(task_mapping).fillna("Unknown")

    # decode each row
    decoded = out["anim_code"].apply(decode_from_code).apply(pd.Series)

    # attach
    for col in ["delay","ease","direction"]:
        out[col] = decoded[col]
    return out

windows_with_anim3 = add_animation_columns(features_task3)
windows_with_anim3.to_csv("/Users/ferolahima/Desktop/gazepoint_project/notebooks/improving models/outputs3/windows_with_animation_details3.csv", index=False)